In [1]:
# ===== Cell 1: parameters =====
import numpy as np

# Problem size
N        = 16                  # tokens
D        = 4                   # embedding dim
NUM_HEAD = 2
HEAD_DIM = D // NUM_HEAD       # 2 (head 1 = dims 0,1 ; head 2 = dims 2,3)
BLOCK    = 2                   # FlashAttention tile size
NUM_BLK  = N // BLOCK          # 8 Q blocks / 8 K,V blocks

# Input magnitude ranges (sign is random +/-)
Q_RANGE = (0.0, 0.5)
K_RANGE = (0.5, 255.0)
V_RANGE = (0.0, 0.5)
W_RANGE = (0.0, 0.5)

# TA constant: parameter sqare_root_2 = 32'b00111111101101010000010011110011
# Decode the exact fp32 bit pattern; the double path uses the same value (not math.sqrt(2)).
SQRT2_BITS = 0b00111111101101010000010011110011            # 0x3FB504F3
SQRT2_F32  = np.array(SQRT2_BITS, dtype=np.uint32).view(np.float32)[()]
SQRT2_F64  = float(SQRT2_F32)

SEED    = 1234
PAT_NUM = 1        # number of patterns (one for now)
ERR_TOL = 1e-6     # TA: demo checks |out - golden| < 1e-6 (spec text "10e-7")
# TA: golden is computed with Verilog real (double), no DesignWare -> golden = flash_double

print(f"SQRT2 = 0x{SQRT2_BITS:08X} -> {SQRT2_F64!r}")

SQRT2 = 0x3FB504F3 -> 1.4142135381698608


In [2]:
# ===== Cell 2: generate test data =====
def f32_to_hex(x):
    """float32 value -> 8-digit hex of its IEEE-754 bits"""
    return f"{int(np.array(x, dtype=np.float32).view(np.uint32)):08X}"

def hex_to_f32(h):
    """8-digit hex -> float32 value"""
    return np.array(int(h, 16), dtype=np.uint32).view(np.float32)[()]

rng = np.random.default_rng(SEED)

def rand_f32(shape, rng_lo_hi):
    """random magnitude in [lo, hi] with random sign, rounded to float32"""
    lo, hi = rng_lo_hi
    mag  = rng.uniform(lo, hi, size=shape)
    sign = rng.choice([-1.0, 1.0], size=shape)
    return np.clip(sign * mag, -hi, hi).astype(np.float32)

# One pattern; matrices are row-major so ravel() == raster order (cycle c -> token c//4, dim c%4)
Q = rand_f32((N, D), Q_RANGE)
K = rand_f32((N, D), K_RANGE)
V = rand_f32((N, D), V_RANGE)
W = rand_f32((D, D), W_RANGE)      # out_weight, 16 values in raster order

Q_hex = [f32_to_hex(x) for x in Q.ravel()]
K_hex = [f32_to_hex(x) for x in K.ravel()]
V_hex = [f32_to_hex(x) for x in V.ravel()]
W_hex = [f32_to_hex(x) for x in W.ravel()]

# hex round-trip sanity check
assert all(hex_to_f32(h) == x for h, x in zip(K_hex, K.ravel()))

for name, mat, hx in (("Q", Q, Q_hex), ("K", K, K_hex), ("V", V, V_hex), ("W", W, W_hex)):
    print(f"{name}: shape={mat.shape}, |min|={np.abs(mat).min():.6g}, |max|={np.abs(mat).max():.6g}, "
          f"first 4 = {hx[:4]}")

Q: shape=(16, 4), |min|=0.00156614, |max|=0.49613, first 4 = ['BEFA08FF', 'BE42A904', 'BEEC59DD', '3E05FC8D']
K: shape=(16, 4), |min|=2.21883, |max|=245.4, first 4 = ['42B375D2', 'C3521603', 'C3718FEA', '418C7EB1']
V: shape=(16, 4), |min|=2.30643e-05, |max|=0.490185, first 4 = ['BEA18BC7', 'BB8A3A30', 'BDE2FA68', '3DDE1924']
W: shape=(4, 4), |min|=0.00163104, |max|=0.460341, first 4 = ['3EEB4462', '3EEBB1D9', '3E031AC3', '3E8940E3']


In [3]:
# ===== Cell 3: computation =====
f32 = np.float32

# ---------- 1. FlashAttention, float64 ----------
def flash_head_double(Qh, Kh, Vh):
    """Spec tiled online-softmax for one head (16x2 slices), float64."""
    out = np.zeros((N, HEAD_DIM))
    for i in range(NUM_BLK):                         # outer loop: Q block
        q = Qh[i*BLOCK:(i+1)*BLOCK]
        m = np.full(BLOCK, -np.inf)                  # running row max
        l = np.zeros(BLOCK)                          # running row sum
        O = np.zeros((BLOCK, HEAD_DIM))              # running O_acc
        for j in range(NUM_BLK):                     # inner loop: K/V block, ascending
            k = Kh[j*BLOCK:(j+1)*BLOCK]
            v = Vh[j*BLOCK:(j+1)*BLOCK]
            S = (q @ k.T) / SQRT2_F64                # 2x2 score tile
            m_new = np.maximum(m, S.max(axis=1))     # always finite
            alpha = np.exp(m - m_new)                # first tile: exp(-inf) = 0 -> no NaN
            P = np.exp(S - m_new[:, None])
            l = alpha * l + P.sum(axis=1)
            O = alpha[:, None] * O + P @ v
            m = m_new
        out[i*BLOCK:(i+1)*BLOCK] = O / l[:, None]   # Head_out = O_acc / l
    return out

# ---------- 2. FlashAttention, every op rounded to float32 ----------
# Fixed evaluation order (no BLAS):
#   2-term dot = (a0*b0) + (a1*b1), rowsum = e0 + e1
# Note: numpy's float32 exp may differ from DesignWare DW_fp_exp by ~1 ulp.
def flash_head_fp32(Qh, Kh, Vh):
    """Same algorithm as flash_head_double, scalar float32 arithmetic."""
    out = np.zeros((N, HEAD_DIM), dtype=np.float32)
    for i in range(NUM_BLK):
        for r in range(BLOCK):                       # Q rows are independent
            q = Qh[i*BLOCK + r]
            m = f32(-np.inf)                         # first tile: alpha = exp(-inf) = 0,
            l = f32(0.0)                             #   0*0 + x == x exactly, so this equals
            O = [f32(0.0)] * HEAD_DIM                #   special-casing tile 0 (and is NaN-free)
            for j in range(NUM_BLK):
                t0, t1 = j*BLOCK, j*BLOCK + 1        # the two tokens of K/V block j
                s0 = (q[0]*Kh[t0, 0] + q[1]*Kh[t0, 1]) / SQRT2_F32
                s1 = (q[0]*Kh[t1, 0] + q[1]*Kh[t1, 1]) / SQRT2_F32
                m_new = max(m, max(s0, s1))
                alpha = np.exp(m - m_new)
                p0 = np.exp(s0 - m_new)
                p1 = np.exp(s1 - m_new)
                l = alpha*l + (p0 + p1)
                O = [alpha*O[d] + (p0*Vh[t0, d] + p1*Vh[t1, d]) for d in range(HEAD_DIM)]
                m = m_new
            for d in range(HEAD_DIM):
                out[i*BLOCK + r, d] = O[d] / l
    return out

# ---------- 3. Standard (non-tiled) attention, float64 ----------
def standard_head_double(Qh, Kh, Vh):
    """Full 16x16 score matrix, row softmax with max subtraction."""
    S = (Qh @ Kh.T) / SQRT2_F64
    P = np.exp(S - S.max(axis=1, keepdims=True))
    P = P / P.sum(axis=1, keepdims=True)
    return P @ Vh

# ---------- multi-head concat + linear layer ----------
def concat_heads(head_fn, Qm, Km, Vm, dtype):
    """head 1 -> dims 0,1 ; head 2 -> dims 2,3"""
    cat = np.zeros((N, D), dtype=dtype)
    for h in range(NUM_HEAD):
        sl = slice(h*HEAD_DIM, (h+1)*HEAD_DIM)
        cat[:, sl] = head_fn(Qm[:, sl], Km[:, sl], Vm[:, sl])
    return cat

# ASSUMPTION (pending TA confirmation): spec figure says Concat x Out_weight^T,
#   Final[t][k] = sum_c Concat[t][c] * W[k][c]
def linear_double(cat, Wm):
    return cat @ Wm.T

def linear_fp32(cat, Wm):
    """left-to-right: ((c0*w0 + c1*w1) + c2*w2) + c3*w3, float32"""
    out = np.zeros((N, D), dtype=np.float32)
    for t in range(N):
        for k in range(D):
            acc = cat[t, 0]*Wm[k, 0] + cat[t, 1]*Wm[k, 1]
            for c in range(2, D):
                acc = acc + cat[t, c]*Wm[k, c]
            out[t, k] = acc
    return out

# ---------- 4. optional torch cross-check ----------
def torch_sdpa_double(Qm, Km, Vm, Wm):
    import torch
    import torch.nn.functional as F
    cat = np.zeros((N, D))
    for h in range(NUM_HEAD):
        sl = slice(h*HEAD_DIM, (h+1)*HEAD_DIM)
        q, k, v = (torch.from_numpy(np.ascontiguousarray(x[:, sl])).unsqueeze(0) for x in (Qm, Km, Vm))
        o = F.scaled_dot_product_attention(q, k, v, scale=1.0/SQRT2_F64)   # divide by TA sqrt2
        cat[:, sl] = o.squeeze(0).numpy()
    return linear_double(cat, Wm)

# ---------- run ----------
Q64, K64, V64, W64 = (x.astype(np.float64) for x in (Q, K, V, W))

out_flash_double = linear_double(concat_heads(flash_head_double, Q64, K64, V64, np.float64), W64)
out_flash_fp32   = linear_fp32(concat_heads(flash_head_fp32, Q, K, V, np.float32), W)
out_std_double   = linear_double(concat_heads(standard_head_double, Q64, K64, V64, np.float64), W64)

try:
    out_torch = torch_sdpa_double(Q64, K64, V64, W64)
except ImportError:
    out_torch = None
    print("torch not installed -> skip SDPA cross-check")

# 64 outputs in raster order
golden_double = out_flash_double.ravel()
golden_fp32   = out_flash_fp32.ravel()
golden_hex    = [f32_to_hex(x) for x in golden_fp32]
print("done")

done


In [4]:
# ===== Cell 4: results =====
err_fp32 = np.abs(golden_fp32.astype(np.float64) - golden_double)
err_std  = np.abs(golden_double - out_std_double.ravel())

print(f"{'idx':>3} {'tok':>3} {'dim':>3}  {'flash_double':>22}  {'flash_fp32':>16}  {'fp32_hex':>8}  "
      f"{'|fp32-double|':>13}  {'|flash-std|':>11}")
for c in range(N*D):
    print(f"{c:3d} {c//D:3d} {c%D:3d}  {golden_double[c]:22.15e}  {golden_fp32[c]:16.9e}  {golden_hex[c]:>8}  "
          f"{err_fp32[c]:13.3e}  {err_std[c]:11.3e}")

print()
all_finite = all(np.isfinite(x).all() for x in (out_flash_double, out_flash_fp32, out_std_double))
print(f"all outputs finite (no NaN/Inf)      : {all_finite}")
print(f"golden                               : flash_double (TA uses Verilog real)")
print(f"max |flash_fp32 - flash_double|      : {err_fp32.max():.3e}  (idx {err_fp32.argmax()})")
print(f"max |flash_double - standard_double| : {err_std.max():.3e}")
if out_torch is not None:
    print(f"max |flash_double - torch_sdpa|      : {np.abs(golden_double - out_torch.ravel()).max():.3e}")
else:
    print("torch cross-check                    : skipped (torch not installed)")
print(f"linear layer orientation             : Final = Concat @ W^T (assumed, pending TA confirmation)")

if err_fp32.max() < ERR_TOL:
    print(f"PASS: fp32 model is within {ERR_TOL:g} of the double golden on all 64 outputs")
else:
    n_bad = int((err_fp32 >= ERR_TOL).sum())
    print(f"FAIL: fp32 model exceeds {ERR_TOL:g} on {n_bad}/64 outputs (max {err_fp32.max():.3e})")

idx tok dim            flash_double        flash_fp32  fp32_hex  |fp32-double|  |flash-std|
  0   0   0  -4.087639219321196e-02  -4.087634385e-02  BD276DF4      4.835e-08    1.388e-17
  1   0   1   5.598502900450914e-02   5.598501489e-02  3D65508B      1.412e-08    0.000e+00
  2   0   2  -3.557518827993911e-02  -3.557518870e-02  BD11B74A      4.164e-10    0.000e+00
  3   0   3  -9.107641868319062e-02  -9.107639641e-02  BDBA8643      2.228e-08    1.388e-17
  4   1   0   1.691387618770268e-01   1.691387445e-01  3E2D32B5      1.740e-08    2.776e-17
  5   1   1  -5.813690573906443e-02  -5.813691020e-02  BD6E20F8      4.461e-09    2.082e-17
  6   1   2   4.181628680828073e-02   4.181627929e-02  3D2B478C      7.516e-09    1.388e-17
  7   1   3   1.555269390703582e-01   1.555269361e-01  3E1F4274      3.016e-09    2.776e-17
  8   2   0  -4.551228382405623e-01  -4.551228583e-01  BEE905DD      2.005e-08    0.000e+00
  9   2   1  -8.033000408376492e-02  -8.032999933e-02  BDA4840E      4.756e-09  